## 1. Business Understanding

The drone delivery company wants to increase revenue by recommending products to customers based on their purchase history. The company focuses on 20 internal product groups and wants to find which product groups act as triggers for buying items from other product groups.

The goal is to use association rule mining to find interesting relationships between product groups and use those rules in a recommender system or product-bundling strategy.

## 2. Data Understanding

The dataset `drone_prod_groups.csv` contains sales data. Each row is a transaction. Columns `Prod1` ... `Prod20` are binary variables indicating whether at least one product from that group was purchased in the transaction.

- `ID`: transaction ID
- `Prod1` ... `Prod20`: 1 = purchased, 0 = not purchased

The dataset has 100000 rows and 21 columns. There are no missing values, and all product columns are integers.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

df = pd.read_csv('drone_prod_groups.csv')
df.head()

,ID,Prod1,Prod2,Prod3,Prod4,Prod5,Prod6,Prod7,Prod8,Prod9,...,Prod11,Prod12,Prod13,Prod14,Prod15,Prod16,Prod17,Prod18,Prod19,Prod20
0,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1
1,2,0,1,0,0,0,0,0,0,1,...,0,0,0,0,1,1,1,1,1,1
2,3,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,1
3,4,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,1
4,5,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,1,1


In [3]:
print('Dataset shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

Dataset shape: (100000, 21)

Data types:
ID         int64
Prod1      int64
 Prod2     int64
 Prod3     int64
 Prod4     int64
 Prod5     int64
 Prod6     int64
 Prod7     int64
 Prod8     int64
 Prod9     int64
 Prod10    int64
 Prod11    int64
 Prod12    int64
 Prod13    int64
 Prod14    int64
 Prod15    int64
 Prod16    int64
 Prod17    int64
 Prod18    int64
 Prod19    int64
 Prod20    int64
dtype: object

Missing values:
ID         0
Prod1      0
 Prod2     0
 Prod3     0
 Prod4     0
 Prod5     0
 Prod6     0
 Prod7     0
 Prod8     0
 Prod9     0
 Prod10    0
 Prod11    0
 Prod12    0
 Prod13    0
 Prod14    0
 Prod15    0
 Prod16    0
 Prod17    0
 Prod18    0
 Prod19    0
 Prod20    0
dtype: int64

Duplicate rows: 0


## Data breakdown

Rows are transactions and values indicate whether a product group was purchased. Value `1` means at least one product from the group was purchased, and `0` means no product from the group was purchased.

No missing values need to be filled. All values are binary, so they can be converted to boolean values. Scaling is not necessary.

## 3. Data Preparation

Drop the `ID` column and convert `1` / `0` values to `True` / `False` for association rule mining.

In [4]:
df = df.drop(columns='ID')
df = df.astype(bool)
df.head()

,Prod1,Prod2,Prod3,Prod4,Prod5,Prod6,Prod7,Prod8,Prod9,Prod10,Prod11,Prod12,Prod13,Prod14,Prod15,Prod16,Prod17,Prod18,Prod19,Prod20
0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True
1,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,True,True,True,True,True
2,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True
3,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True
4,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,True


## 4. Modeling

Use the Apriori algorithm to find frequent itemsets. First, test different `min_support` values to find a good balance between the number of itemsets and rule quality.

In [5]:
support_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
results = []
for min_supp in support_values:
    frequent_itemsets = apriori(df, min_support=min_supp, use_colnames=True)
    rules_conf = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)
    rules_lift = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    results.append({
        'min_support': min_supp,
        'frequent_itemsets': len(frequent_itemsets),
        'rules_conf_0.5': len(rules_conf),
        'rules_lift_1.0': len(rules_lift)
    })
support_test = pd.DataFrame(results)
support_test

,min_support,frequent_itemsets,rules_conf_0.5,rules_lift_1.0
0,0.001,1564,888,15216
1,0.002,903,551,6598
2,0.005,339,133,1506
3,0.010,170,77,472
4,0.020,97,31,208
5,0.050,19,5,6


Based on the support test, `min_support = 0.01` provides a good balance. It gives enough frequent itemsets and rules without becoming too noisy.

In [6]:
frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.10998,frozenset({Prod1})
1,0.13098,frozenset({ Prod2})
2,0.03271,frozenset({ Prod3})
3,0.03585,frozenset({ Prod4})
4,0.10459,frozenset({ Prod5})
...,...,...
165,0.02030,"frozenset({ Prod15, Prod20, Prod19})"
166,0.02203,"frozenset({ Prod16, Prod20, Prod19})"
167,0.02052,"frozenset({ Prod20, Prod19, Prod18})"
168,0.01101,"frozenset({ Prod12, Prod20, Prod19, Prod5})"


Next, test different confidence thresholds to see how many rules remain.

In [7]:
confidence_values = [0.3, 0.4, 0.425, 0.45, 0.5, 0.525, 0.55, 0.575, 0.6, 0.7, 0.8]
results = []
for min_conf in confidence_values:
    rules_test = association_rules(frequent_itemsets, metric='confidence', min_threshold=min_conf)
    results.append({'min_confidence': min_conf, 'rules': len(rules_test)})
confidence_test = pd.DataFrame(results)
confidence_test

,min_confidence,rules
0,0.300,91
1,0.400,89
2,0.425,81
3,0.450,77
4,0.500,77
5,0.525,76
6,0.550,76
7,0.575,63
8,0.600,60
9,0.700,32


Next, test different lift thresholds to find a suitable minimum lift.

In [8]:
lift_values = [1, 1.2, 1.5, 2, 3, 4, 4.25, 4.5, 4.75]
results = []
for min_lift in lift_values:
    rules_test = association_rules(frequent_itemsets, metric='lift', min_threshold=min_lift)
    results.append({'min_lift': min_lift, 'rules': len(rules_test)})
lift_test = pd.DataFrame(results)
lift_test

,min_lift,rules
0,1.00,472
1,1.20,426
2,1.50,170
3,2.00,170
4,3.00,170
5,4.00,160
6,4.25,128
7,4.50,82
8,4.75,38


## 5. Evaluation

The selected thresholds are:

- `support >= 0.01`
- `confidence >= 0.55`
- `lift >= 4.0`

These values give sufficiently confident rules while still providing enough recommendations for a recommender system.

In [9]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.0)
rules = rules[
    (rules['support'] >= 0.01) &
    (rules['confidence'] >= 0.55) &
    (rules['lift'] >= 4.0)
]
rules = rules.sort_values(by=['confidence', 'lift', 'support'], ascending=False)
rules_report = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules_report

,antecedents,consequents,support,confidence,lift
242,"frozenset({ Prod15, Prod2})",frozenset({ Prod9}),0.01843,0.946584,4.767967
392,"frozenset({ Prod15, Prod20})",frozenset({ Prod9}),0.02119,0.945560,4.762807
461,"frozenset({ Prod15, Prod20, Prod19})",frozenset({ Prod9}),0.01919,0.945320,4.761599
356,"frozenset({ Prod12, Prod15})",frozenset({ Prod9}),0.02173,0.941508,4.742396
314,"frozenset({ Prod15, Prod7})",frozenset({ Prod9}),0.01895,0.940914,4.739403
...,...,...,...,...,...
367,"frozenset({ Prod9, Prod14})",frozenset({ Prod15}),0.01969,0.565155,4.757197
355,"frozenset({ Prod9, Prod12})",frozenset({ Prod15}),0.02173,0.565150,4.757151
325,"frozenset({ Prod9, Prod8})",frozenset({ Prod15}),0.02195,0.563833,4.746065
130,frozenset({ Prod9}),frozenset({ Prod15}),0.11145,0.561376,4.725388


In [10]:
rules_report['antecedents'] = rules_report['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_report['consequents'] = rules_report['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_report

,antecedents,consequents,support,confidence,lift
242,"Prod15, Prod2",Prod9,0.01843,0.946584,4.767967
392,"Prod15, Prod20",Prod9,0.02119,0.945560,4.762807
461,"Prod15, Prod19, Prod20",Prod9,0.01919,0.945320,4.761599
356,"Prod12, Prod15",Prod9,0.02173,0.941508,4.742396
314,"Prod15, Prod7",Prod9,0.01895,0.940914,4.739403
...,...,...,...,...,...
367,"Prod14, Prod9",Prod15,0.01969,0.565155,4.757197
355,"Prod12, Prod9",Prod15,0.02173,0.565150,4.757151
325,"Prod8, Prod9",Prod15,0.02195,0.563833,4.746065
130,Prod9,Prod15,0.11145,0.561376,4.725388


### Evaluation observations

The strongest rules involve `Prod9` and `Prod15`. For example:

- `{Prod2, Prod15} -> {Prod9}` with confidence about `0.947` and lift about `4.77`
- `{Prod20, Prod15} -> {Prod9}` with confidence about `0.946` and lift about `4.76`
- `{Prod15} -> {Prod9}` with confidence about `0.938` and lift about `4.73`
- `{Prod9} -> {Prod15}` with confidence about `0.561` and lift about `4.73`
- `{Prod20} -> {Prod19}` with confidence about `0.911` and lift about `4.42`
- `{Prod19} -> {Prod20}` with confidence about `0.653` and lift about `4.42`

`Prod9` has strong cross-selling relationships with `Prod15`, `Prod2`, `Prod20`, `Prod19`, `Prod7`, and `Prod12`. The pair `Prod19` / `Prod20` also has a strong relationship.

## 6. Deployment / Conclusion

The company should use Apriori with approximately:

- `min_support = 0.01`
- `min_confidence = 0.55`
- `min_lift = 4.0`

Recommended actions:

1. When a customer has `Prod15`, `Prod2`, `Prod20`, `Prod19`, `Prod7`, or `Prod12` in their cart, recommend `Prod9`.
2. When a customer has `Prod9` in their cart, recommend `Prod15`.
3. Use `Prod19` and `Prod20` as a cross-sell or bundle pair.
4. Use the discovered rules for frequently-bought-together recommendations or product bundles.
5. Re-run the analysis periodically with new sales data and re-tune the support, confidence, and lift thresholds.

The model does not provide recommendations for every possible product combination, but it provides sufficiently confident rules for a useful recommender system. Lowering support too much would increase the number of rules but reduce reliability.